# RQ2 cross-subnet gradient interaction probe

A standalone, read-only mechanistic diagnostic optimized for **T4 x2**. It uses the common **Uniform seed-3 epoch-100** checkpoint, splits 16 fixed minibatches into two disjoint 8-batch GPU shards, and uses the exact frozen Geometry-Dynamic p=1 / matched-compute Resource-Dynamic marginals. It never invokes or resumes seed-4 training, never takes an optimizer step, and never opens the test split.

In [ ]:
import os, subprocess, sys, json, shutil, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1'], check=True)
import torch
assert torch.cuda.device_count() == 2, f'Select Kaggle T4 x2; detected {torch.cuda.device_count()} GPU(s)'
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('Diagnostic GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])

## Locate frozen development evidence and the common checkpoint

In [ ]:
import importlib
import rq2_anchor_placement, rq2_cross_subnet_interaction
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_cross_subnet_interaction = importlib.reload(rq2_cross_subnet_interaction)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
REFERENCE_INPUT = Path('/kaggle/input/notebooks/dyhngg/rq2-seed-3-4-5')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
assert REFERENCE_INPUT.exists(), f'Attach Uniform seed-3 output: {REFERENCE_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(RQ2_INPUT, '/kaggle/working/materialized-rq2-interaction')
CHECKPOINT = rq2_cross_subnet_interaction.find_uniform_seed3_checkpoint(REFERENCE_INPUT, '/kaggle/working/materialized-uniform-seed3')
CONFIG_PATH = RQ2_ROOT/'resolved_config.yaml'
print('Frozen development root:', RQ2_ROOT)
print('Common reference checkpoint:', CHECKPOINT)
print('Config:', CONFIG_PATH)

## Reproduce the exact frozen seed-3 policy marginals
This is CPU-only policy reconstruction from development geometry seeds 0/1/2. No accuracy is used to construct either policy.

In [ ]:
import rq2_theory_allocation_probe, rq2_probabilistic_support
rq2_theory_allocation_probe = importlib.reload(rq2_theory_allocation_probe)
rq2_probabilistic_support = importlib.reload(rq2_probabilistic_support)
RUN_DIR = Path('/kaggle/working/rq2-cross-subnet-interaction')
THEORY_ROOT = RUN_DIR/'frozen_policy'/'theory-allocation-probe'
PREVIEW_ROOT = RUN_DIR/'frozen_policy'/'probabilistic-support-preview'
theory = rq2_theory_allocation_probe.run_theory_probe(RQ2_ROOT, THEORY_ROOT, waiting_steps=2_000_000)
preview = rq2_probabilistic_support.build_support_policy(RQ2_ROOT, PREVIEW_ROOT, pi_min=1e-8)
assert theory['training_authorized'] is False and preview['training_authorized'] is False
assert all(value for key, value in theory.items() if key.endswith('_pass'))
assert all(preview['assertions'].values())
GEOMETRY_MARGINALS = THEORY_ROOT/'policy_family_marginals.csv'
RESOURCE_MARGINALS = PREVIEW_ROOT/'support_allocation_marginals.csv'
print('Geometry policy: p=1 rows from', GEOMETRY_MARGINALS)
print('Resource policy: matched-compute column from', RESOURCE_MARGINALS)

## Run the 16-batch read-only probe on T4 x2
GPU 0 processes global batches 0–7 and the one-step transfer; GPU 1 processes batches 8–15 concurrently. Each worker calibrates identical BN banks, then enters eval mode. Teacher logits are detached and parameter/buffer hashes are checked before and after.

In [ ]:
observed_candidates = sorted(Path('/kaggle/input').rglob('dynamic_pilot_width_comparison.csv'))
OBSERVED = observed_candidates[0] if len(observed_candidates) == 1 else None
if OBSERVED is None:
    print('Observed seed-3 comparison not uniquely available; gradient outputs will still be complete.')
else:
    print('Post-hoc observed comparison:', OBSERVED)
import scripts.run_cross_subnet_interaction_t4x2 as interaction_runner
interaction_runner = importlib.reload(interaction_runner)
started = time.perf_counter()
metadata = interaction_runner.run_t4x2_probe(
    checkpoint=CHECKPOINT,
    config_path=CONFIG_PATH,
    geometry_marginals_path=GEOMETRY_MARGINALS,
    resource_marginals_path=RESOURCE_MARGINALS,
    output_dir=RUN_DIR,
    dataset_root='/kaggle/working/data',
    observed_comparison_path=OBSERVED,
    gpu_ids=[0,1],
    num_batches=16,
)
metadata['git_commit'] = GIT_COMMIT
(RUN_DIR/'metadata.json').write_text(json.dumps(metadata, indent=2)+'\n')
print(f"Probe completed in {(time.perf_counter()-started)/60:.1f} minutes")
print(json.dumps(metadata, indent=2))

## Inspect the decision-facing outputs

In [ ]:
import pandas as pd
from IPython.display import display, Image
effect = pd.read_csv(RUN_DIR/'policy_expected_effect.csv')
display(effect)
display(effect.loc[effect.width.isin([0.25,0.40,0.75,0.80,0.85,0.90,0.95,1.00])])
transfer = pd.read_csv(RUN_DIR/'one_step_transfer.csv')
print('One-step sign agreement:', transfer.sign_match.mean())
display(transfer.groupby('source_width', as_index=False).sign_match.mean())
display(Image(filename=str(RUN_DIR/'interaction_heatmap.png')))
display(Image(filename=str(RUN_DIR/'policy_effect_by_width.png')))

## Validate and export the small diagnostic bundle

In [ ]:
required = [
    'metadata.json','gradient_cosine_matrix.csv','gradient_dot_matrix.csv',
    'gradient_norms.csv','gradient_norms_by_batch.csv','policy_expected_effect.csv',
    'policy_expected_effect_by_batch.csv','interaction_heatmap.png',
    'policy_effect_by_width.png','one_step_transfer.csv',
    'fixed_training_subset_ids.csv','gram_matrices.npy','fold_assignment.csv',
]
missing = [name for name in required if not (RUN_DIR/name).is_file() or (RUN_DIR/name).stat().st_size == 0]
assert not missing, f'Missing diagnostic artifacts: {missing}'
saved = json.loads((RUN_DIR/'metadata.json').read_text())
assert saved['execution'] == 'two_gpu_disjoint_batch_shards' and saved['gpu_workers'] == 2
assert saved['training_performed'] is False and saved['optimizer_steps'] == 0
assert saved['test_used'] is False and saved['independent_of_seed4_training'] is True
assert saved['weights_unchanged'] is True and saved['bn_buffers_unchanged_during_probe'] is True
bundle_path = Path('/kaggle/working/rq2-cross-subnet-interaction.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in RUN_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(RUN_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path